In [3]:
from pathlib import Path
import pandas as pd
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config.config import MERGED_DATA

DATA_DIR = Path("../data")
DATA_TICKER = DATA_DIR / "processed" / "layoffs_with_tickers.csv"

# Load layoff data and financial features.
layoffs = pd.read_csv(DATA_TICKER)
features_df = pd.read_csv(MERGED_DATA["MERGED_OUTPUT_CSV_PATH"])

# Company in features_df is a ticker, so labels must be keyed by Ticker, not Company_name.
layoffs["date"] = pd.to_datetime(layoffs["date"], errors="coerce")
layoffs = layoffs.dropna(subset=["Ticker", "date"])
layoffs["quarter"] = layoffs["date"].dt.to_period("Q").astype(str)

layoff_events = set(
    zip(
        layoffs["Ticker"].astype(str),
        layoffs["quarter"].astype(str),
    )
)

feature_quarters = pd.PeriodIndex(features_df["quarter"], freq="Q")
label_df = pd.DataFrame(
    {
        "same_quarter": feature_quarters.astype(str),
        "next_quarter": (feature_quarters + 1).astype(str),
    }
)

same_quarter_pairs = list(
    zip(features_df["company"].astype(str), label_df["same_quarter"])
)
next_quarter_pairs = list(
    zip(features_df["company"].astype(str), label_df["next_quarter"])
)

label_df["layoff_same_quarter"] = (
    pd.Series(same_quarter_pairs).isin(layoff_events).astype(int).values
)
label_df["layoff_next_quarter"] = (
    pd.Series(next_quarter_pairs).isin(layoff_events).astype(int).values
)
label_df["layoff_same_or_next_quarter"] = (
    (label_df["layoff_same_quarter"] == 1)
    | (label_df["layoff_next_quarter"] == 1)
).astype(int)

# Default target for modeling. Use layoff_next_quarter if you want stricter prediction.
label_df["layoff"] = label_df["layoff_same_or_next_quarter"]

dataset = pd.concat([features_df.reset_index(drop=True), label_df], axis=1)

print(dataset["layoff"].value_counts())
print(dataset["layoff"].value_counts(normalize=True))
print(dataset[["layoff_same_quarter", "layoff_next_quarter", "layoff_same_or_next_quarter", "layoff"]].sum())

print(
    "Positive companies:",
    dataset.loc[dataset["layoff"].eq(1), "company"].nunique(),
)

dataset.to_csv(
    MERGED_DATA["LABELED_OUTPUT_CSV_PATH"],
    index=False,
)


layoff
0    12002
1       45
Name: count, dtype: int64
layoff
0    0.996265
1    0.003735
Name: proportion, dtype: float64
layoff
0    12002
1       45
Name: count, dtype: int64
